# **LangChain으로 문서 불러오기 (DocumentLoader)**

## 실습 소개

LangChain의 Document Loader는 다양한 문서 형식을 불러와서 <b>LangChain에서 처리 가능한 구조화된 문서 객체(Document)</b>로 변환해주는 구성요소입니다. 예를 들어, PDF, 텍스트 파일, 웹 페이지, CSV 등 다양한 입력 소스를 LangChain 내부에서 사용할 수 있도록 **통합된 문서 형식**으로 바꿔줍니다.

LangChain의 `Document`는 기본적으로 다음 두 가지 속성을 가집니다.

- `.page_content`: 문서의 본문 텍스트
- `.metadata`: 문서 출처, 위치 등의 메타데이터

이 실습에서는 LangChain의 대표적인 로더들을 활용하여 서로 다른 종류의 데이터 소스를 **동일한 방식으로 로드하고 확인하는 방법**을 학습합니다.

## 학습 목표

- 다양한 유형의 문서를 LangChain에서 처리 가능한 형태로 불러오는 방법 익히기
- 각 Document Loader의 기본 사용법 습득
- 불러온 문서에서 `.page_content`와 `.metadata`를 활용해 내용 확인

## 대표적인 DataLoader

| 로더 이름             | 설명                                     | 대표 사용 예시                          |
|----------------------|------------------------------------------|------------------------------------------|
| `TextLoader`         | 일반 텍스트 파일 로딩                     | `.txt` 문서 한 개 불러오기                |
| `PyPDFLoader`        | PDF 파일의 각 페이지를 Document로 분리    | 리포트, 논문, 스캔 문서 등                |
| `WebBaseLoader`      | 웹 페이지를 불러와 HTML을 파싱            | 블로그 글, 기사, 문서 포털 등             |
| `CSVLoader`          | CSV 파일에서 특정 컬럼의 내용을 로드      | 뉴스 제목, 리뷰, 설명문 등 구조화된 데이터 |

---


### TextLoader

LangChain의 `TextLoader`는 `.txt` 파일을 불러오는 데 사용됩니다.

In [3]:
from langchain.document_loaders import TextLoader

ModuleNotFoundError: No module named 'langchain.document_loaders'

질문을 위한 텍스트 파일을 준비하겠습니다.

`TextLoader('경로명')`을 사용하여 지정된 경로의 파일을 불러옵니다. 이 불러온 텍스트를 `.load()`를 활용하여 `Document` 객체 리스트로 변환합니다.

In [ ]:
loader = TextLoader("example.txt", encoding="utf-8")
documents = loader.load()

불러온 텍스트는 문서 1개당 `Document` 하나로 래핑됩니다. 불러온 `example.txt`를 살펴보겠습니다.

In [ ]:
documents[0]

LangChain에서는 한 줄씩 나눠서 로딩하거나, 나중에 `TextSplitter`와 함께 사용할 수도 있습니다.

파일을 한 줄씩 읽어 문서 여러 개로 나누는 커스텀 로더도 만들 수 있습니다.

In [ ]:
from langchain_core.documents import Document

with open("example.txt", "r", encoding="utf-8") as f:
    lines = f.readlines()

documents = [Document(page_content=line.strip()) for line in lines if line.strip()]

In [ ]:
documents

### PyPDFLoader

`PyPDFLoader`는 PDF 파일을 불러와 LangChain이 사용할 수 있는 `Document` 형식으로 변환하는 로더입니다.

이 로더는 **PDF의 각 페이지를 하나의 Document로 분리**해 줍니다.  
예를 들어 10페이지짜리 PDF를 로드하면, `10개의 Document`가 생성됩니다.

PDF는 학술 논문, 백서, 리포트, 계약서 등 **구조화된 문서의 대표적인 형태**이기 때문에 LangChain 기반 요약, 질의 응답 시스템에 자주 사용됩니다.

> 사용 시 내부적으로 `PyPDF2` 또는 `pdfplumber` 같은 PDF 파서를 활용합니다.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

`example.pdf`라는 파일을 불러오기 위한 `loader` 객체를 생성합니다. 이때 PDF는 페이지 단위로 분리되어 각각의 Document 객체로 변환됩니다.

`.load()` 메서드를 호출하면 PDF가 LangChain의 `Document` 리스트로 반환됩니다.

각 페이지는 하나의 `Document` 객체가 되며 `.page_content`로 본문에 접근할 수 있습니다.

In [ ]:
loader = PyPDFLoader("example.pdf")

documents = loader.load()

첫 번째 페이지의 본문 내용을 출력합니다.

In [ ]:
print(documents[0].page_content)

### WebBaseLoader

`WebBaseLoader`는 웹 페이지의 내용을 불러와 `Document`로 변환하는 로더입니다.  
웹 페이지의 HTML을 불러온 뒤, 텍스트만 추출하여 문서 객체로 만듭니다.

블로그, 뉴스, 위키 문서 등 **웹 기반 정보 소스**를 LLM의 입력으로 활용하고자 할 때 유용합니다.

> 내부적으로 `requests`와 `beautifulsoup4`를 사용하여 HTML을 크롤링하고 정제합니다.


In [ ]:
from langchain_community.document_loaders import WebBaseLoader

`https://en.wikipedia.org/wiki/LangChain`이라는 URL의 HTML 콘텐츠를 불러옵니다.

`.load()`를 호출하면 웹 페이지 내용을 LangChain의 `Document` 객체로 변환해 반환합니다.

내부적으로 `BeautifulSoup`을 활용해 텍스트를 추출합니다.

In [ ]:
loader = WebBaseLoader("https://en.wikipedia.org/wiki/LangChain")

documents = loader.load()

로드된 웹 페이지 중 <b>본문 일부(앞 500자)</b>를 출력해 확인합니다.

`WebBaseLoader`는 웹 페이지의 HTML을 불러온 후 BeautifulSoup으로 텍스트를 추출합니다. 이때 일부 웹페이지는 `<script>` 나 `<style>`, 또는 깨진 구조로 인해 알아보기 어려울 수 있습니다. 따라서 아래 코드는 텍스트를 전처리하여 출력합니다.

In [ ]:
import re

text = documents[0].page_content
clean_text = re.sub(r'\s+', ' ', text).strip()
print(clean_text[:500])

### CSVLoader

`CSVLoader`는 CSV 파일의 특정 컬럼을 불러와 `Document`로 변환하는 로더입니다.

CSV 파일은 일반적으로 표 형태로 데이터를 저장하는 구조이며, 리뷰, 기사 제목, 텍스트 응답, 질문 등 구조화된 정보를 많이 포함합니다.

- `source_column="내용"`처럼 컬럼명을 지정하면 해당 열의 값만 불러옵니다.
- 각 행은 하나의 `Document`가 됩니다.

> 텍스트 데이터가 열 단위로 존재하는 CSV는 **텍스트 기반 LLM 태스크의 주요 소스**로 활용됩니다.


In [ ]:
from langchain_community.document_loaders.csv_loader import CSVLoader

`example.csv` 파일의 "Team" 컬럼만 추출해 각 행을 문서로 로드하도록 설정합니다. 이때 `source_column`에 지정한 컬럼만을 `page_content`로 사용합니다.

`.load()`를 실행하면 CSV의 각 행이 하나의 `Document` 객체로 생성됩니다.

In [ ]:
loader = CSVLoader(file_path="example.csv", source_column="Team")

documents = loader.load()

첫 번째 CSV 행의 내용을 출력합니다.

In [ ]:
print(documents[0].page_content)